In [1]:
from src.utils.qdrant_store import QdrantStore

In [5]:
# Embedding models you want to try
embedding_models = [
#   "sentence-transformers/all-MiniLM-L6-v2",       
#    "facebook/bart-large",                          
#    "sentence-transformers/paraphrase-mpnet-base-v2",
    "microsoft/codebert-base",
#    "Salesforce/codet5-base",
]

# Distance metrics
distances = ["cosine"]

# Base collection name
base_collection_name = "rag_collection"

# Loop through all model + distance combinations
for model in embedding_models:
    for dist in distances:
        # Create a unique collection name for each combo
        collection_name = f"{base_collection_name}_{model.split('/')[-1]}_{dist}"

        print(f"\n=== Creating store for {collection_name} ===")
        print(dist)
        # Initialize QdrantStore
        store = QdrantStore(
            model_name=model,
            collection_name=collection_name,
            qdrant_url="http://localhost:6333",
            api_key="@lmafa12",
            neo4j_uri="bolt://localhost:7687",
            neo4j_auth=("neo4j", "password"),
            distance_type=dist
        )

        # Clear if exists (fresh start each time)
        store.clear_collection()

        # Add Issues
        store.add_from_neo4j(node_label="ISSUE", text_property="issue_title", id_property="ID", metadata_type="issue_title")
        print(f"status after adding issue titles: {store.get_collection_info()}")
        store.add_from_neo4j(node_label="ISSUE", text_property="issue_body", id_property="ID", metadata_type="issue_body")
        print(f"status after adding issue bodies: {store.get_collection_info()}")

        # Add Functions
        store.add_from_neo4j(node_label="FUNCTION", text_property="function_code", id_property="ID", metadata_type="function_code")
        print(f"status after adding function codes: {store.get_collection_info()}")
        store.add_from_neo4j(node_label="FUNCTION", text_property="combinedName", id_property="ID", metadata_type="function_name")
        print(f"status after adding function names: {store.get_collection_info()}")

        # Add PRs
        store.add_from_neo4j(node_label="PR", text_property="pr_title", id_property="ID", metadata_type="pr_title")
        print(f"status after adding pr titles: {store.get_collection_info()}")
        store.add_from_neo4j(node_label="PR", text_property="pr_body", id_property="ID", metadata_type="pr_body")
        print(f"status after adding pr bodies: {store.get_collection_info()}")

        # Add imports from Neo4j to Qdrant
        store.add_from_neo4j(node_label="IMPORT", text_property="import_name",id_property="ID", metadata_type="import_name")
        print(f"status after adding import names: {store.get_collection_info()}")
        # Add clusters from Neo4j to Qdrant
        store.add_from_neo4j(node_label="CLUSTER", text_property="summary",id_property="ID", metadata_type="semantic_cluster")
        # Print final collection info
        print(store.get_collection_info())


No sentence-transformers model found with name microsoft/codebert-base. Creating a new one with mean pooling.



=== Creating store for rag_collection_codebert-base_cosine ===
cosine
Successfully connected to Qdrant
Creating collection 'rag_collection_codebert-base_cosine'.
status after adding issue titles: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=None indexed_vectors_count=0 points_count=2459 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size

In [2]:
store = QdrantStore(
            model_name="microsoft/codebert-base",
            collection_name="rag_collection_codebert-base_cosine",
            qdrant_url="http://localhost:6333",
            api_key="@lmafa12",
            neo4j_uri="bolt://localhost:7687",
            neo4j_auth=("neo4j", "password"),
            distance_type="cosine"
        )

d:\EricssonCodeGraph\hgb-rag-cqa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No sentence-transformers model found with name microsoft/codebert-base. Creating a new one with mean pooling.
d:\EricssonCodeGraph\hgb-rag-cqa\src\utils\qdrant_store.py:28: UserWarning: Api key is used with an insecure connection.
  self.client = QdrantClient(url=qdrant_url, api_key=api_key)


Successfully connected to Qdrant
Collection 'rag_collection_codebert-base_cosine' already exists.


In [19]:
first = store.search_with_scores("What kinds of regression models and related methods are available in scikit-learn for beginners to try?", top_k=5)
first

[(Document(metadata={'type': 'issue_body', 'node_id': 23779, 'doc_id': '6cf4989b-14a7-43e5-a26d-c49d5b33d531', 'chunk_size': 128, '_id': 'b2d22de9-1ed8-48a0-9165-2f090d76a87a', '_collection_name': 'rag_collection_codebert-base_cosine'}, page_content='I have no example using only scikit-learn, as you need custom types and classifiers / transformers to trigger it'),
  0.9892116),
 (Document(metadata={'type': 'issue_body', 'node_id': 23779, 'doc_id': 'f5d4ea37-a956-4463-a89a-6c2148506062', 'chunk_size': 256, '_id': '6c2047f2-0ef2-45d3-9a22-c5fabbe63090', '_collection_name': 'rag_collection_codebert-base_cosine'}, page_content='I have no example using only scikit-learn, as you need custom types and classifiers / transformers to trigger it'),
  0.98921144),
 (Document(metadata={'type': 'function_code', 'node_id': 7290, 'doc_id': 'ce1addad-c650-4473-bba6-02df7e0c9693', 'chunk_size': 128, '_id': '25d12335-4546-4498-86b2-e20c36d72d86', '_collection_name': 'rag_collection_codebert-base_cosine'}

In [20]:
second = store.search_with_scores("How to implement a binary search tree in Python?", top_k=5)
second

[(Document(metadata={'type': 'issue_body', 'node_id': 11799, 'doc_id': '39b39216-cfa8-4bfb-8fb9-f843659eb445', 'chunk_size': 128, '_id': 'd692782b-839e-4616-95db-7a59f444bc8e', '_collection_name': 'rag_collection_codebert-base_cosine'}, page_content='I also tried to implement a class inheriting the LDA to negate the score, but the score is internally used in the LDA class'),
  0.98333454),
 (Document(metadata={'type': 'issue_body', 'node_id': 25647, 'doc_id': '0712f79b-a50d-42de-8ffc-68edad68449d', 'chunk_size': 128, '_id': 'd43a43fc-e52a-455a-a4a4-8072dc93a444', '_collection_name': 'rag_collection_codebert-base_cosine'}, page_content='2) Apply the pd.dropna() function on only the relevant feature while looping. My current work around is to do this in'),
  0.9833087),
 (Document(metadata={'type': 'issue_body', 'node_id': 17639, 'doc_id': '41fb9c15-1477-4fb6-8ccc-e0b8f500f3a5', 'chunk_size': 256, '_id': '145b3035-4318-400b-8faa-f8f610d85d47', '_collection_name': 'rag_collection_codebert

In [22]:
merged = first+second
for doc,_ in merged:
    print(doc.page_content[:200], "\n---")

I have no example using only scikit-learn, as you need custom types and classifiers / transformers to trigger it 
---
I have no example using only scikit-learn, as you need custom types and classifiers / transformers to trigger it 
---
however some input validation code in scikit-learn needs to work both 
---
metric in scikit-learn that is more appropriate for that use case? 
---
Since sklearn's tf-idf is widely used, I think it should reflect standard behavior/definitions in the IR and NLP literature, 
---
I also tried to implement a class inheriting the LDA to negate the score, but the score is internally used in the LDA class 
---
2) Apply the pd.dropna() function on only the relevant feature while looping. My current work around is to do this in 
---
in scikit-learn that is more appropriate for that use case? 
---
metric in scikit-learn that is more appropriate for that use case? 
---
Note: we probably need to improve our CI to also test for the installation of the generated tarbal

In [4]:
store.client.delete_collection("rag_collection_codebert-base_cosine")

True